In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

# Ensure project root is on sys.path when notebook is executed from notebooks/
_PROJECT_ROOT = Path(os.path.abspath("")).parent
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))
os.chdir(_PROJECT_ROOT)  # make OUT_DIR relative paths resolve correctly

import hdf5plugin  # registers bitshuffle and other HDF5 filters (Eiger4M needs it)
import h5py
import matplotlib.pyplot as plt
import numpy as np

# Real detector files (pre-assembled 2D frames, one frame per index along axis-0)
DETECTOR_FILES: dict[str, str] = {
    "AGIPD":       "/Users/gketawal/Desktop/detector-images/AGIPD.cxi",
    "JUNGFRAU_4M": "/Users/gketawal/Desktop/detector-images/JUNFRAU.h5",
    "ePix10k":     "/Users/gketawal/Desktop/detector-images/epix10k.cxi",
    "Eiger4M":     "/Users/gketawal/Desktop/detector-images/Eiger4M.h5",
}

# HDF5 data keys confirmed by scripts/probe_hdf5.py (2026-05-21)
HDF5_KEYS: dict[str, str] = {
    "AGIPD":       "entry_1/data_1/data",
    "JUNGFRAU_4M": "entry_0000/instrument/Simulator/data",
    "ePix10k":     "entry_1/data_1/data",
    "Eiger4M":     "entry/data/data",
}

WINDOWS = [3, 9, 15, 31]
OUT_DIR = Path("docs/figures/lcn_ablation")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
from skimage.transform import resize as sk_resize
from src.preprocessing.normalize import gcn, lcn

TARGET_SIZE = (224, 224)


def load_and_preprocess(detector: str, lcn_window: int) -> np.ndarray:
    """Load first assembled frame from detector file and apply GCN → LCN → resize.

    All four detector files store pre-assembled 2D frames stacked along axis-0.
    We index frame[0] to get a single 2D array and run normalization directly,
    bypassing Reborn geometry assembly (frames are already assembled by CrystFEL
    or the beamline DAQ and do not need further tiling).
    """
    path = DETECTOR_FILES[detector]
    key = HDF5_KEYS[detector]
    with h5py.File(path, "r") as f:
        frame = f[key][0].astype(np.float64)   # shape: (H, W)

    frame_gcn = gcn(frame)
    frame_lcn = lcn(frame_gcn, window=lcn_window)
    resized = sk_resize(frame_lcn, TARGET_SIZE, anti_aliasing=True, preserve_range=True)
    return resized.astype(np.float32)

In [ ]:
fig, axes = plt.subplots(
    len(DETECTOR_FILES), len(WINDOWS),
    figsize=(4 * len(WINDOWS), 3 * len(DETECTOR_FILES)),
)

for row_idx, detector in enumerate(DETECTOR_FILES):
    images = [load_and_preprocess(detector, w) for w in WINDOWS]
    vmin = min(img.min() for img in images)
    vmax = max(img.max() for img in images)

    for col_idx, (w, img) in enumerate(zip(WINDOWS, images)):
        ax = axes[row_idx, col_idx]
        ax.imshow(img, cmap="viridis", vmin=vmin, vmax=vmax, origin="lower")
        ax.set_title(f"window={w}", fontsize=9)
        if col_idx == 0:
            ax.set_ylabel(detector, fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])

plt.suptitle("LCN Window Ablation — all four detectors", fontsize=12)
plt.tight_layout()
plt.savefig(OUT_DIR / "all_detectors_lcn_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved all_detectors_lcn_comparison.png")

In [ ]:
for detector in DETECTOR_FILES:
    fig, axes = plt.subplots(1, len(WINDOWS), figsize=(4 * len(WINDOWS), 3))
    images = [load_and_preprocess(detector, w) for w in WINDOWS]
    vmin = min(img.min() for img in images)
    vmax = max(img.max() for img in images)
    for ax, w, img in zip(axes, WINDOWS, images):
        ax.imshow(img, cmap="viridis", vmin=vmin, vmax=vmax, origin="lower")
        ax.set_title(f"window={w}", fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
    fig.suptitle(detector, fontsize=11)
    plt.tight_layout()
    out = OUT_DIR / f"{detector}_lcn_comparison.png"
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved {out}")

## Decision

After visual inspection:

- **Chosen window size:** _fill in after review_
- **Rationale:** _fill in after review_
- **Action:** Update `LCN_WINDOW_DEFAULT` in `src/preprocessing/normalize.py` if not 9.